# N-BEATS

N-BEATS is a deep model made of stacked fully-connected blocks. Each block predicts a
backcast and a forecast, the leftover (residual) goes to the next block. It is a
global model: one model learns from all Store+Dept series at once.

WMAE is measured on the real validation rows (from `get_real_validation`), so it is
directly comparable to the tree models. Experiments are logged to the shared DagsHub.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import yaml
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from src.features.nn_preprocessing import build_long_df, temporal_split, get_real_validation, FREQ
from src.models.nbeats_pipeline import build_nbeats
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")

# Shared DagsHub tracking.
dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("NBEATS_Training")

# Load the model config.
with open("configs/nbeats_config.yaml") as f:
    config = yaml.safe_load(f)
params = config["model"]["params"]
split_date = config["data"]["split_date"]
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-09 20:01:00,263	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-07-09 20:01:00,814	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/09 20:01:05 INFO mlflow.tracking.fluent: Experiment with name 'NBEATS_Training' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape)

train: (421570, 5)


## Run 1: Data Preparation

Build the long panel for training and the real validation rows for scoring.

In [4]:
with mlflow.start_run(run_name="NBEATS_Data_Prep"):
    # Training data: gap-filled long panel.
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    train_df, valid_df, horizon = temporal_split(Y_df, split_date)
    # Real validation rows for a fair, comparable WMAE.
    real_valid = get_real_validation(train_raw, stores, features, split_date)

    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", split_date)
    mlflow.log_param("n_series", int(Y_df["unique_id"].nunique()))
    mlflow.log_param("horizon_weeks", int(horizon))
    mlflow.log_param("real_valid_rows", int(len(real_valid)))

    print("series:", Y_df["unique_id"].nunique(), "| horizon:", horizon,
          "| real valid rows:", len(real_valid))

series: 3331 | horizon: 43 | real valid rows: 127438
🏃 View run NBEATS_Data_Prep at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/5/runs/5780c7b38faa4be5a0d35300b0013d17
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/5


## WMAE on real validation rows

Join the forecast to the real validation rows and use the shared `calculate_wmae`.

In [5]:
def evaluate_on_real(forecast_df, real_valid, model_col="NBEATS"):
    merged = real_valid.merge(
        forecast_df[["unique_id", "ds", model_col]], on=["unique_id", "ds"], how="inner",
    )
    preds = merged[model_col].clip(lower=0)  # sales can't be negative
    wmae = calculate_wmae(merged["y"], preds, merged["IsHoliday"])
    return wmae, merged

## Run 2: Training

Train N-BEATS from the config and score it on the real validation rows.
(Earlier exploration: interpretable stacks gave ~2,768 WMAE on the gap-filled set and
the generic stacks in this config won, so generic is kept.)

In [6]:
with mlflow.start_run(run_name="NBEATS_Training"):
    mlflow.log_params({k: str(v) for k, v in params.items()})

    nf = build_nbeats(horizon, params, freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_on_real(fcst, real_valid, "NBEATS")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"NBEATS WMAE (real validation): {wmae:,.2f}")

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 5.2 M  | train
-------------------------------------------------------
5.2 M     Trainable params
0         Non-trainable params
5.2 M     Total params
20.732    Total estimated model params size (MB)
58        Modules in train mode
0         Modules in eval mode


Epoch 0:  96%|█████████▌| 100/104 [00:38<00:01,  2.63it/s, v_num=0, train_loss_step=81.10] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1:  92%|█████████▏| 96/104 [00:43<00:03,  2.21it/s, v_num=0, train_loss_step=134.0, train_loss_epoch=109.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2:  88%|████████▊ | 92/104 [00:40<00:05,  2.28it/s, v_num=0, train_loss_step=16.90, train_loss_epoch=117.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3:  85%|████████▍ | 88/104 [00:32<00:05,  2.69it/s, v_num=0, train_loss_step=456.0, train_loss_epoch=118.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4:  81%|████████  | 84/104 [00:32<00:07,  2.56it/s, v_num=0, train_loss_step=111.0, train_loss_epoch=140.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [

`Trainer.fit` stopped: `max_steps=1000` reached.


Epoch 9: 100%|██████████| 64/64 [00:29<00:00,  2.14it/s, v_num=0, train_loss_step=67.70, train_loss_epoch=113.0]


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|██████████| 104/104 [00:01<00:00, 66.16it/s]
NBEATS WMAE (real validation): 2,277.43
🏃 View run NBEATS_Training at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/5/runs/445a8d7f78374fb5b5ad6ae371965920
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/5


## Run 3: Best Pipeline (retrain on all data, save)

Retrain on the full history and save the model for the joint inference/ensemble step.

In [7]:
with mlflow.start_run(run_name="NBEATS_Best_Pipeline"):
    mlflow.log_params({k: str(v) for k, v in params.items()})

    nf_best = build_nbeats(horizon, params, freq=FREQ)
    nf_best.fit(df=Y_df[["unique_id", "ds", "y"]])  # full history

    save_path = "saved_models/nbeats_nf"
    os.makedirs(save_path, exist_ok=True)
    nf_best.save(path=save_path, overwrite=True, save_dataset=False)
    mlflow.log_artifacts(save_path, artifact_path="nbeats_nf")
    print("Saved NBEATS model to", save_path)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 5.2 M  | train
-------------------------------------------------------
5.2 M     Trainable params
0         Non-trainable params
5.2 M     Total params
20.732    Total estimated model params size (MB)
58        Modules in train mode
0         Modules in eval mode


Epoch 0:  95%|█████████▌| 100/105 [00:47<00:02,  2.12it/s, v_num=2, train_loss_step=37.40]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1:  90%|█████████ | 95/105 [00:40<00:04,  2.37it/s, v_num=2, train_loss_step=23.70, train_loss_epoch=77.60] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2:  86%|████████▌ | 90/105 [00:38<00:06,  2.33it/s, v_num=2, train_loss_step=17.50, train_loss_epoch=161.0]   
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3:  81%|████████  | 85/105 [00:33<00:07,  2.52it/s, v_num=2, train_loss_step=44.20, train_loss_epoch=62.20]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4:  76%|███████▌  | 80/105 [00:30<00:09,  2.61it/s, v_num=2, train_loss_step=19.70, train_loss_epoch=68.90] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00

`Trainer.fit` stopped: `max_steps=1000` reached.


Epoch 9: 100%|██████████| 55/55 [00:25<00:00,  2.19it/s, v_num=2, train_loss_step=15.70, train_loss_epoch=92.30]
Saved NBEATS model to saved_models/nbeats_nf
🏃 View run NBEATS_Best_Pipeline at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/5/runs/f9b9aa13c4d34e449d40cc9ab66f1286
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/5


## Notes

- WMAE here is on the real validation rows, so it is comparable to the tree models.
- Model params live in `configs/nbeats_config.yaml`; the model is built in
  `src/models/nbeats_pipeline.py`.
- The saved model feeds the joint ensemble/inference step.